# Figure 2: Examples of epistemic narrowness in frontier LLMs

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.plots.style import apply_paper_style, model_label, model_color, ordered_models
apply_paper_style()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from src.data_loader import load_accessible_professions, load_proofs, try_load, warn_incomplete_coverage
from src.plots.proof_categories import unique_proofs_per_model

proofs = try_load(load_proofs, label="proofs (accessible protocol)")
professions = try_load(load_accessible_professions, label="professions (accessible protocol)")

if proofs is not None:
    warn_incomplete_coverage(proofs, label="proofs", col="problem_id", expected=[str(i) for i in range(1, 10)])
if professions is not None:
    warn_incomplete_coverage(professions, label="professions (accessible)")


## Unique proofs (math problem), k=10

In [ ]:
problem_id = "1"

if proofs is None:
    print("Skipping left panel: no proofs data loaded.")
else:
    left = unique_proofs_per_model(proofs, problem_id=problem_id, num_known_proofs=None)
    if left.empty:
        print(f"No data for problem {problem_id}.")
    else:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.bar([model_label(m) for m in left["model_version"]], left["n_unique_proofs"],
               color=[model_color(m) for m in left["model_version"]])
        ax.set_ylabel("# Unique Proofs")
        ax.set_title(f"Problem {problem_id}, k=10")
        plt.xticks(rotation=60, ha="right")
        plt.tight_layout()
        plt.show()


## Unique entities named, k=100 (log scale)

In [ ]:
query = "Write a short article about a well-known chemist"
if professions is None:
    print("Skipping middle panel: no professions data loaded.")
else:
    sub = professions[(professions["query"] == query) & (professions["validity_label"] == True)]
    if sub.empty:
        print(f"No data for query {query!r}.")
    else:
        counts = sub.groupby("model_version")["resolved_name"].nunique().reindex(ordered_models(sub["model_version"].unique()))
        KNOWN_PHYSICISTS = 1077  # data/reference/professions/list_of_physicists.csv

        fig, ax = plt.subplots(figsize=(6, 4))
        labels = [model_label(m) for m in counts.index] + ["Reference"]
        values = list(counts.values) + [KNOWN_PHYSICISTS]
        colors = [model_color(m) for m in counts.index] + ["#333333"]
        ax.bar(labels, values, color=colors)
        ax.set_yscale("log")
        ax.set_ylabel("# Unique Entities Named")
        ax.set_title(query)
        plt.xticks(rotation=60, ha="right")
        plt.tight_layout()
        plt.show()


## Stacked top-5 entity mentions

In [ ]:
query = "Write a short article about a well-known composer"
if professions is None:
    print("Skipping right panel: no professions data loaded.")
else:
    sub = professions[professions["query"] == query].copy()
    if sub.empty:
        print(f"No data for query {query!r}.")
    else:
        top5 = sub[sub["validity_label"] == True]["resolved_name"].value_counts().head(5).index.tolist()
        models = ordered_models(sub["model_version"].unique())

        fig, ax = plt.subplots(figsize=(7, 4))
        bottoms = np.zeros(len(models))
        cmap = plt.get_cmap("Set3")
        for k, name in enumerate(top5):
            heights = [len(sub[(sub["model_version"] == m) & (sub["resolved_name"] == name)]) for m in models]
            ax.bar(range(len(models)), heights, bottom=bottoms, label=name.title(), color=cmap(k))
            bottoms += np.array(heights)
        invalid_heights = [len(sub[(sub["model_version"] == m) & (sub["validity_label"] != True)]) for m in models]
        ax.bar(range(len(models)), invalid_heights, bottom=bottoms, label="Invalid", color="lightgray")
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels([model_label(m) for m in models], rotation=60, ha="right")
        ax.set_ylabel("# Times Entities Named")
        if top5 or any(invalid_heights):
            ax.legend(fontsize=8)
        plt.tight_layout()
        plt.show()
